In [128]:
import heapq
import math
from collections import Counter
import os
import random
import string
import time

In [129]:
random.seed(42)


def gen_high_redundancy(size=50000):
    words = [
        "the",
        "and",
        "is",
        "in",
        "of",
        "to",
        "a",
        "that",
        "it",
        "was",
        "for",
        "on",
        "are",
        "as",
        "with",
        "his",
        "they",
        "at",
        "be",
        "this",
        "data",
        "compression",
        "the",
        "the",
        "the",
    ]
    text = []
    for _ in range(size // 4):
        text.append(random.choice(words))
    return " ".join(text)[:size]


def gen_natural_english():
    passage = (
        "The quick brown fox jumps over the lazy dog. Data compression is a fundamental "
        "concept in computer science and information theory. The goal of compression is to "
        "represent information using fewer bits than the original representation. Lossless "
        "compression algorithms allow the original data to be perfectly reconstructed from "
        "the compressed data. The Lempel Ziv family of algorithms form the basis of many "
        "popular compression formats used today. Huffman coding is an entropy coding algorithm "
        "that assigns shorter codes to more frequent symbols. The compression ratio measures "
        "how effectively an algorithm reduces the size of the data. Information theory provides "
        "theoretical bounds on how much data can be compressed. The entropy of a source is the "
        "minimum number of bits needed per symbol for lossless compression. Dictionary based "
        "methods exploit repeated patterns and substrings within the data. Natural language text "
        "exhibits high redundancy due to the statistical structure of language. Common words and "
        "phrases appear with high frequency and can be encoded efficiently. "
    )
    return (passage * 60)[:50000]


def gen_low_redundancy(size=50000):
    chars = string.ascii_letters + string.digits + "+/="
    return "".join(random.choice(chars) for _ in range(size))


os.makedirs("datasets", exist_ok=True)

datasets = {
    "dataset1_high_redundancy.txt": gen_high_redundancy(),
    "dataset2_natural_english.txt": gen_natural_english(),
    "dataset3_low_redundancy.txt": gen_low_redundancy(),
}

for fname, content in datasets.items():
    path = f"datasets/{fname}"
    with open(path, "w") as f:
        f.write(content)
    size = os.path.getsize(path)
    unique = len(set(content))
    print(
        f"Created {path}: {len(content)} chars, {size} bytes, {unique} unique symbols"
    )

Created datasets/dataset1_high_redundancy.txt: 50000 chars, 50000 bytes, 18 unique symbols
Created datasets/dataset2_natural_english.txt: 50000 chars, 50000 bytes, 36 unique symbols
Created datasets/dataset3_low_redundancy.txt: 50000 chars, 50000 bytes, 65 unique symbols


In [130]:
# HUFFMAN ENCODING
# --------------

In [131]:
class HuffmanNode:
    __slots__ = ("symbol", "freq", "left", "right")

    def __init__(self, symbol, freq, left=None, right=None):
        self.symbol = symbol
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq

    def __eq__(self, other):
        return self.freq == other.freq


def build_huffman_tree(data: str):
    freq = Counter(data)

    if len(freq) == 1:
        sym = next(iter(freq))
        leaf = HuffmanNode(sym, freq[sym])
        return HuffmanNode(None, freq[sym], left=leaf)

    heap = [HuffmanNode(sym, cnt) for sym, cnt in freq.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = HuffmanNode(None, left.freq + right.freq, left, right)
        heapq.heappush(heap, merged)

    return heap[0] if heap else None

def generate_codes(root: HuffmanNode) -> dict:
    codes = {}

    def traverse(node, path):
        if node is None:
            return
        if node.symbol is not None:
            codes[node.symbol] = path or "0"
            return
        traverse(node.left, path + "0")
        traverse(node.right, path + "1")

    traverse(root, "")
    return codes


def huffman_compress(data: str):
    if not data:
        return b"", {}, 0, None

    root = build_huffman_tree(data)
    codes = generate_codes(root)

    bitstring = "".join(codes[ch] for ch in data)

    padding = (8 - len(bitstring) % 8) % 8
    bitstring += "0" * padding

    compressed = bytearray()
    for i in range(0, len(bitstring), 8):
        compressed.append(int(bitstring[i : i + 8], 2))

    return bytes(compressed), codes, padding, root


def huffman_decompress(
    compressed: bytes, root: HuffmanNode, padding: int, original_length: int
) -> str:
    if not compressed or root is None:
        return ""

    bitstring = "".join(format(byte, "08b") for byte in compressed)
    if padding:
        bitstring = bitstring[:-padding]

    output = []
    current = root

    for bit in bitstring:
        current = current.left if bit == "0" else current.right

        if current is None:
            break

        if current.symbol is not None:
            output.append(current.symbol)
            current = root

            if len(output) == original_length:
                break

    return "".join(output)


In [132]:
def serialize_tree(root: HuffmanNode) -> str:
    if root is None:
        return ""
    if root.symbol is not None:
        return f"L{ord(root.symbol):03d}"
    return "I" + serialize_tree(root.left) + serialize_tree(root.right)


def compute_entropy(data: str) -> float:
    n = len(data)
    freq = Counter(data)
    entropy = 0.0

    for count in freq.values():
        p = count / n
        if p > 0:
            entropy -= p * math.log2(p)

    return entropy


def compute_average_code_length(data: str, codes: dict) -> float:
    n = len(data)
    freq = Counter(data)
    return sum((freq[sym] / n) * len(code) for sym, code in codes.items())


def compute_snr(original_bytes: bytes, recovered_bytes: bytes):
    n = min(len(original_bytes), len(recovered_bytes))
    if n == 0:
        return float("inf")

    signal_power = sum(x * x for x in original_bytes[:n])
    noise_power = sum(
        (x - y) ** 2 for x, y in zip(original_bytes[:n], recovered_bytes[:n])
    )

    if noise_power == 0:
        return float("inf")

    return 10 * math.log10(signal_power / noise_power)


def compute_energy_compaction(data: str, k: int = 3):
    freq = Counter(data)
    n = len(data)

    if n == 0:
        return 0.0

    probs = sorted((count / n for count in freq.values()), reverse=True)
    return sum(probs[:k])


def evaluate_huffman(data: str):
    original_bytes = data.encode("utf-8")
    original_size = len(original_bytes)

    compressed, codes, padding, root = huffman_compress(data)
    compressed_size = len(compressed)

    tree_overhead = len(codes) * 2
    total_size = compressed_size + tree_overhead

    recovered = huffman_decompress(compressed, root, padding, len(data))
    assert recovered == data

    compression_ratio = original_size / total_size if total_size else float("inf")
    bits_per_symbol = (total_size * 8) / len(data)
    entropy = compute_entropy(data)
    redundancy = 1.0 - (entropy / 8.0)
    avg_code_len = compute_average_code_length(data, codes)

    orig_vals = list(original_bytes)
    recv_vals = list(recovered.encode("utf-8"))
    n = min(len(orig_vals), len(recv_vals))

    mse = (
        sum((a - b) ** 2 for a, b in zip(orig_vals[:n], recv_vals[:n])) / n
        if n
        else 0.0
    )

    psnr = float("inf") if mse == 0 else 10 * math.log10((255**2) / mse)

    snr = compute_snr(original_bytes, recovered.encode("utf-8"))
    energy_compaction = compute_energy_compaction(data, k=3)

    return {
        "algorithm": "Huffman",
        "original_size_bytes": original_size,
        "compressed_size_bytes": total_size,
        "compression_ratio": round(compression_ratio, 4),
        "bits_per_symbol": round(bits_per_symbol, 4),
        "entropy": round(entropy, 4),
        "redundancy": round(redundancy, 4),
        "mse": round(mse, 6),
        "psnr": round(psnr, 4) if psnr != float("inf") else "inf",
        "avg_code_length": round(avg_code_len, 4),
        "snr": round(snr, 4) if snr != float("inf") else "inf",
        "energy_compaction_top3": round(energy_compaction, 4),
        "alphabet_size": len(codes),
    }

In [133]:
# LZ77 Compression

In [134]:
def lz77_compress(data: str, window_size: int = 255, lookahead_size: int = 15):
    tokens = []
    i = 0
    n = len(data)

    while i < n:
        best_offset = 0
        best_length = 0

        search_start = max(0, i - window_size)
        search_buf = data[search_start:i]
        lookahead = data[i : i + lookahead_size]

        for length in range(min(lookahead_size, n - i), 0, -1):
            pattern = data[i : i + length]
            pos = search_buf.rfind(pattern)
            if pos != -1:
                best_offset = len(search_buf) - pos
                best_length = length
                break

        next_pos = i + best_length
        if next_pos < n:
            next_char = ord(data[next_pos])
        else:
            next_char = 0

        tokens.append((best_offset, best_length, next_char))
        i += best_length + 1

    return tokens

def lz77_decompress(tokens):
    output = []

    for offset, length, next_char in tokens:
        if length > 0 and offset > 0:
            start = len(output) - offset
            for k in range(length):
                output.append(output[start + k])

        if next_char != 0:
            output.append(chr(next_char))

    return "".join(output)

In [135]:
def tokens_to_bytes(tokens, window_size=255, lookahead_size=15):
    offset_bits = math.ceil(math.log2(window_size + 1))
    length_bits = math.ceil(math.log2(lookahead_size + 1))

    bits = []
    for offset, length, next_char in tokens:
        bits.extend(int(b) for b in format(offset, f"0{offset_bits}b"))
        bits.extend(int(b) for b in format(length, f"0{length_bits}b"))
        bits.extend(int(b) for b in format(next_char, "08b"))

    while len(bits) % 8 != 0:
        bits.append(0)

    result = bytearray()
    for i in range(0, len(bits), 8):
        byte = 0
        for bit in bits[i : i + 8]:
            byte = (byte << 1) | bit
        result.append(byte)

    return bytes(result)


def compute_entropy(data: str) -> float:
    n = len(data)
    freq = Counter(data)
    entropy = 0.0

    for count in freq.values():
        p = count / n
        if p > 0:
            entropy -= p * math.log2(p)

    return entropy


def compute_snr(original_bytes: bytes, recovered_bytes: bytes):
    n = min(len(original_bytes), len(recovered_bytes))
    if n == 0:
        return float("inf")

    signal_power = sum(x * x for x in original_bytes[:n])
    noise_power = sum(
        (x - y) ** 2 for x, y in zip(original_bytes[:n], recovered_bytes[:n])
    )

    if noise_power == 0:
        return float("inf")

    return 10 * math.log10(signal_power / noise_power)


def compute_energy_compaction_lz77(tokens, data_length: int):
    if data_length == 0:
        return 0.0

    matched = sum(length for (_, length, _) in tokens)
    return matched / data_length


def evaluate_lz77(data: str, window_size: int = 255, lookahead_size: int = 15):
    original_bytes = data.encode("utf-8")
    original_size = len(original_bytes)

    tokens = lz77_compress(data, window_size, lookahead_size)

    compressed = tokens_to_bytes(tokens, window_size, lookahead_size)
    compressed_size = len(compressed)

    recovered = lz77_decompress(tokens)
    assert recovered == data

    compression_ratio = (
        original_size / compressed_size if compressed_size else float("inf")
    )
    bits_per_symbol = (compressed_size * 8) / len(data)
    entropy = compute_entropy(data)
    redundancy = 1.0 - (entropy / 8.0)

    orig_vals = list(original_bytes)
    recv_vals = list(recovered.encode("utf-8"))
    n = min(len(orig_vals), len(recv_vals))

    mse = (
        sum((a - b) ** 2 for a, b in zip(orig_vals[:n], recv_vals[:n])) / n
        if n
        else 0.0
    )
    psnr = float("inf") if mse == 0 else 10 * math.log10((255**2) / mse)

    snr = compute_snr(original_bytes, recovered.encode("utf-8"))
    energy_compaction = compute_energy_compaction_lz77(tokens, len(data))

    return {
        "algorithm": "LZ77",
        "original_size_bytes": original_size,
        "compressed_size_bytes": compressed_size,
        "compression_ratio": round(compression_ratio, 4),
        "bits_per_symbol": round(bits_per_symbol, 4),
        "entropy": round(entropy, 4),
        "redundancy": round(redundancy, 4),
        "mse": round(mse, 6),
        "psnr": round(psnr, 4) if psnr != float("inf") else "inf",
        "snr": round(snr, 4) if snr != float("inf") else "inf",
        "energy_compaction": round(energy_compaction, 4),
        "num_tokens": len(tokens),
        "window_size": window_size,
        "lookahead_size": lookahead_size,
    }


In [136]:
# deflate (LZSS + Huffman)
import math

import heapq
from collections import Counter

def deflate_compress(data: str, window_size: int = 32768, lookahead_size: int = 258):
    tokens = lz77_compress(data, window_size, lookahead_size)

    symbols = []

    for offset, length, next_char in tokens:
        if offset == 0 and length <= 2:
            symbols.append(next_char)
        else:
            symbols.append(256 + length)
            symbols.append(512 + offset)
            if next_char != 0:
                symbols.append(next_char)

    # --- build Huffman ---
    freq = Counter(symbols)

    heap = [[freq[sym], [sym, ""]] for sym in freq]
    heapq.heapify(heap)

    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]:
            pair[1] = "0" + pair[1]
        for pair in hi[1:]:
            pair[1] = "1" + pair[1]
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])

    codes = dict(heap[0][1:])

    bitstring = "".join(codes[s] for s in symbols)

    padding = (8 - len(bitstring) % 8) % 8
    bitstring += "0" * padding

    compressed = bytearray()
    for i in range(0, len(bitstring), 8):
        compressed.append(int(bitstring[i:i+8], 2))

    return bytes(compressed), codes, padding, None, len(symbols)

def deflate_decompress(compressed, codes, padding, root, original_length,
                       window_size: int = 255, lookahead_size: int = 15):

    reverse = {v: k for k, v in codes.items()}

    bitstring = "".join(format(b, "08b") for b in compressed)
    if padding:
        bitstring = bitstring[:-padding]

    symbols = []
    cur = ""

    for bit in bitstring:
        cur += bit
        if cur in reverse:
            symbols.append(reverse[cur])
            cur = ""
        if len(symbols) == original_length:
            break

    output = []
    i = 0

    while i < len(symbols):
        sym = symbols[i]

        if sym < 256:
            output.append(chr(sym))
            i += 1
        else:
            length = sym - 256
            offset = symbols[i+1] - 512
            i += 2

            start = len(output) - offset
            for j in range(length):
                output.append(output[start + j])

            if i < len(symbols) and symbols[i] < 256:
                output.append(chr(symbols[i]))
                i += 1

    return "".join(output)

In [137]:
def test_deflate(data: str):
    compressed, codes, padding, root, original_length = deflate_compress(data)
    decompressed = deflate_decompress(compressed, codes, padding, root, original_length)

    assert decompressed == data, "Decompression failed!"

    original_bits = len(data) * 8
    compressed_bits = len(compressed) * 8 - padding

    compression_ratio = original_bits / compressed_bits
    space_saving = (1 - 1/compression_ratio) * 100

    lz77_tokens = lz77_compress(data)
    literal_count = sum(1 for o, l, _ in lz77_tokens if o == 0 and l == 0)
    match_count = len(lz77_tokens) - literal_count

    avg_match_length = (
        sum(l for o, l, _ in lz77_tokens if l > 0) / match_count
        if match_count > 0 else 0
    )

    print("=== Compression Metrics ===")
    print(f"Original size      : {original_bits} bits ({len(data)} bytes)")
    print(f"Compressed size    : {compressed_bits} bits ({len(compressed)} bytes)")
    print(f"Compression ratio  : {compression_ratio:.4f}")
    print(f"Space saving       : {space_saving:.2f}%")

    print("\n=== Token Stats ===")
    print(f"Total tokens       : {len(lz77_tokens)}")
    print(f"Literals           : {literal_count}")
    print(f"Matches            : {match_count}")
    print(f"Avg match length   : {avg_match_length:.2f}")

test_deflate("""
On a quiet morning in early summer, when the light had only just begun to warm the edges of the day, there was a particular stillness that seemed to settle over everything. It was not the kind of silence that feels empty or hollow, but rather one that feels full, as if the world itself were pausing to take a breath before beginning again. The air was soft and carried the faint scent of damp earth and distant trees. Somewhere far away, a dog barked once, then stopped, as if it too had decided not to disturb the moment.

In a small house at the edge of a narrow road, a young man sat by a window with a cup of tea growing steadily cooler beside him. He had been awake for hours, though he could not have said exactly why. Sleep had come and gone in uneven waves, leaving behind a strange mixture of restlessness and clarity. It was one of those mornings when thoughts felt unusually sharp, when even the smallest ideas seemed to carry a kind of quiet importance.

He watched as the light shifted slowly across the floor, stretching long shadows into shorter ones. Dust motes drifted lazily through the air, turning and spinning in the sunlight like tiny, aimless travelers. There was something calming about watching them, something that made time feel less urgent.

He thought about the many things he had planned to do that day. There were tasks waiting for him, responsibilities that could not be ignored forever. Yet in that moment, none of them seemed particularly pressing. Instead, his mind wandered to memories that surfaced without warning, fragments of conversations, places he had visited, and people he had not seen in years.

Memory has a peculiar way of reshaping itself. Moments that once felt insignificant can become meaningful over time, while others that seemed important fade into the background. He remembered a rainy afternoon from his childhood, standing under a small shelter with a friend, both of them laughing at nothing in particular. He could not recall what they had been laughing about, only that the laughter itself had felt endless.

There is something about simple moments that makes them endure. Perhaps it is because they are not burdened by expectation. They are not planned or forced into existence; they simply happen. And when they do, they leave behind a quiet impression that lingers long after the details have faded.

The young man took a sip of his tea, now lukewarm, and grimaced slightly before setting the cup down again. He considered making another, but decided against it. Instead, he leaned back in his chair and closed his eyes, listening.

The world was waking up. A bicycle passed by on the road outside, its wheels making a soft, rhythmic sound. A few voices could be heard in the distance, indistinct but lively. Somewhere nearby, a door opened and shut. These small sounds formed a kind of gentle rhythm, a reminder that life was continuing all around him.

It struck him then how easy it is to overlook such things. In the rush of daily life, these moments are often lost, dismissed as background noise. Yet when one takes the time to notice them, they reveal a quiet beauty that is always present, waiting to be seen.

He opened his eyes again and stood up, stretching slightly as he moved toward the window. Outside, the sky was a pale blue, with thin clouds drifting slowly across it. The trees along the road swayed gently in the breeze, their leaves catching the light in shifting patterns.

He wondered, not for the first time, how many mornings like this he had missed. How many times had he rushed through the start of the day, focused only on what needed to be done next? It was a strange realization, that something so ordinary could also be so easily forgotten.

There is a tendency in people to seek out the extraordinary, to look for moments that stand out, that can be clearly defined as important or meaningful. But perhaps meaning is not always found in such moments. Perhaps it is found in the quiet, unremarkable parts of life, in the spaces between events.

As he stood there, he felt a subtle shift in his perspective. It was not a dramatic change, not something that could be easily described or explained. It was simply a sense of awareness, a recognition of something that had always been there.

He decided then to step outside.

The air was cooler than he had expected, and he paused for a moment on the doorstep, taking it in. The road stretched out in both directions, empty except for a single figure walking in the distance. The world felt open and unhurried.

He began to walk without any particular destination in mind. Each step felt deliberate, as if he were rediscovering the simple act of moving forward. The ground beneath his feet was uneven in places, small stones shifting slightly as he walked over them.

As he continued along the road, he noticed details he might have otherwise ignored. The texture of the walls of nearby houses, worn and marked by time. The way the sunlight reflected off a small puddle, creating a brief shimmer before fading again. The faint sound of wind passing through tall grass.

There is a certain clarity that comes from slowing down. When one is not rushing from one place to another, there is space to notice things that would otherwise go unseen. It is not that these details are hidden; they are simply overlooked.

He reached a bend in the road and stopped, looking out over a field that stretched beyond the houses. The grass was a deep green, moving in waves as the wind passed through it. In the distance, a few birds rose into the air, their movements quick and unpredictable.

He felt, in that moment, a sense of connection that was difficult to put into words. It was not tied to any specific thought or realization. It was simply a feeling of being present, of being part of something larger.

Time passed, though he did not keep track of it. The sun rose higher, the light becoming stronger, more defined. The stillness of the morning gave way to the activity of the day. More people appeared on the road, cars passing by, voices growing louder.

Eventually, he turned back toward the house.

When he returned, the world felt slightly different, though nothing had really changed. The same room, the same furniture, the same cup of tea still sitting by the window. Yet there was a subtle shift in how he saw it all.

He picked up the cup and took another sip, then smiled faintly. It was still lukewarm, but it no longer seemed to matter.

He sat down again and looked out the window, watching as the day continued to unfold. There were still things to be done, tasks waiting for his attention. But now, they felt less overwhelming, less urgent.

It is easy to become caught up in the idea that everything must be done quickly, efficiently, without pause. But perhaps there is value in slowing down, in allowing time for moments like this to exist.

The young man did not come to any grand conclusions that morning. There was no sudden realization that changed everything. Instead, there was something quieter, something more subtle.

He had simply taken the time to notice.

And in doing so, he had discovered something that had always been there, waiting.

The rest of the day passed as most days do, with its mix of small successes and minor frustrations. There were moments of focus and moments of distraction, conversations that were meaningful and others that were quickly forgotten.

Yet throughout it all, there remained a faint awareness of the morning, a reminder of the stillness, the light, the quiet sense of presence.

It is often said that life is made up of moments, but not all moments are given equal attention. Some are highlighted, remembered, and revisited, while others pass by unnoticed.

But perhaps the unnoticed moments are just as important.

Perhaps they form the foundation upon which everything else is built.

As evening approached and the light began to fade, the young man found himself once again by the window. The sky had changed, shifting from pale blue to deeper shades of orange and purple. The shadows had grown longer again, stretching across the floor in familiar patterns.

He watched as the first lights appeared in the distance, small points of brightness against the dimming sky. The world was settling down, moving toward another kind of stillness.

There was a sense of completion to the day, though nothing in particular had been completed. It was simply the natural rhythm of time, the movement from one state to another.

He thought again about the morning, about the quiet realization that had come to him without effort. It was not something he needed to hold onto tightly or analyze in detail. It was enough to know that it had happened.

He stood up and closed the window, the sound of the outside world becoming softer as it faded away. The room felt warmer now, more enclosed.

As he prepared to end the day, he felt a quiet sense of ease. Not everything was resolved, not everything was clear. But there was a kind of acceptance in that.

Life does not always need to be fully understood. Sometimes, it is enough to simply experience it, to be present in the moment without trying to define it.

And so the day came to an end, as all days do, not with a dramatic conclusion, but with a gradual fading into night.

The stillness returned, different from the stillness of the morning, yet connected to it in a way that was difficult to describe.

Somewhere in that stillness, the young man slept.

And when the next morning came, as it inevitably would, the world would begin again.
""")


def evaluate_deflate(data):
    compressed, codes, padding, root, original_length = deflate_compress(data)
    decompressed = deflate_decompress(compressed, codes, padding, root, original_length)

    assert decompressed == data, "Deflate decompression failed!"

    original_bits = len(data) * 8
    compressed_bits = len(compressed) * 8 - padding

    compression_ratio = original_bits / compressed_bits
    bits_per_symbol = compressed_bits / len(data)

    return {
        "compression_ratio": compression_ratio,
        "bits_per_symbol": bits_per_symbol,
    }

=== Compression Metrics ===
Original size      : 76688 bits (9586 bytes)
Compressed size    : 40298 bits (5038 bytes)
Compression ratio  : 1.9030
Space saving       : 47.45%

=== Token Stats ===
Total tokens       : 2981
Literals           : 51
Matches            : 2930
Avg match length   : 2.25


In [138]:
# LZW Compression

In [139]:
CLEAR_CODE = 256
EOF_CODE = 257
FIRST_CODE = 258
MIN_BITS = 9
MAX_BITS = 16


def lzw_compress(data: str):
    dictionary = {chr(i): i for i in range(256)}
    next_code = FIRST_CODE
    bit_width = MIN_BITS

    codes = [CLEAR_CODE]
    code_widths = [bit_width]

    w = ""

    for char in data:
        wc = w + char
        if wc in dictionary:
            w = wc
        else:
            codes.append(dictionary[w])
            code_widths.append(bit_width)

            if next_code < (1 << MAX_BITS):
                dictionary[wc] = next_code
                next_code += 1

                if next_code > (1 << bit_width) and bit_width < MAX_BITS:
                    bit_width += 1

            w = char

    if w:
        codes.append(dictionary[w])
        code_widths.append(bit_width)

    codes.append(EOF_CODE)
    code_widths.append(bit_width)

    return codes, code_widths


def lzw_decompress(codes, code_widths=None):
    dictionary = {i: chr(i) for i in range(256)}
    next_code = FIRST_CODE

    output = []
    i = 0

    if codes and codes[0] == CLEAR_CODE:
        i = 1

    if i >= len(codes):
        return ""

    first_code = codes[i]
    if first_code == EOF_CODE:
        return ""

    previous = dictionary[first_code]
    output.append(previous)
    i += 1

    while i < len(codes):
        code = codes[i]
        i += 1

        if code == EOF_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {j: chr(j) for j in range(256)}
            next_code = FIRST_CODE
            if i < len(codes):
                first_code = codes[i]
                i += 1
                previous = dictionary[first_code]
                output.append(previous)
            continue

        if code in dictionary:
            entry = dictionary[code]
        elif code == next_code:
            entry = previous + previous[0]
        else:
            raise ValueError(f"Bad LZW code: {code}")

        output.append(entry)

        if next_code < (1 << MAX_BITS):
            dictionary[next_code] = previous + entry[0]
            next_code += 1

        previous = entry

    return "".join(output)

In [140]:
def codes_to_bytes(codes, code_widths):
    bits = []
    for code, width in zip(codes, code_widths):
        bits.extend(int(b) for b in format(code, f"0{width}b"))

    while len(bits) % 8 != 0:
        bits.append(0)

    result = bytearray()
    for i in range(0, len(bits), 8):
        byte = 0
        for bit in bits[i : i + 8]:
            byte = (byte << 1) | bit
        result.append(byte)

    return bytes(result)


def compute_entropy(data: str) -> float:
    n = len(data)
    freq = Counter(data)
    entropy = 0.0

    for count in freq.values():
        p = count / n
        if p > 0:
            entropy -= p * math.log2(p)

    return entropy


def compute_snr(original_bytes: bytes, recovered_bytes: bytes):
    n = min(len(original_bytes), len(recovered_bytes))
    if n == 0:
        return float("inf")

    signal_power = sum(x * x for x in original_bytes[:n])
    noise_power = sum(
        (x - y) ** 2 for x, y in zip(original_bytes[:n], recovered_bytes[:n])
    )

    if noise_power == 0:
        return float("inf")

    return 10 * math.log10(signal_power / noise_power)


def compute_energy_compaction_lzw(codes, data_length: int):
    if data_length == 0:
        return 0.0

    effective_codes = len(codes) - 2
    if effective_codes <= 0:
        return 0.0

    return 1 - (effective_codes / data_length)


def evaluate_lzw(data: str):
    original_bytes = data.encode("utf-8")
    original_size = len(original_bytes)

    codes, code_widths = lzw_compress(data)
    compressed = codes_to_bytes(codes, code_widths)
    compressed_size = len(compressed)

    recovered = lzw_decompress(codes, code_widths)
    assert recovered == data

    compression_ratio = (
        original_size / compressed_size if compressed_size else float("inf")
    )
    bits_per_symbol = (compressed_size * 8) / len(data)
    entropy = compute_entropy(data)
    redundancy = 1.0 - (entropy / 8.0)

    orig_vals = list(original_bytes)
    recv_vals = list(recovered.encode("utf-8"))
    n = min(len(orig_vals), len(recv_vals))

    mse = (
        sum((a - b) ** 2 for a, b in zip(orig_vals[:n], recv_vals[:n])) / n
        if n
        else 0.0
    )
    psnr = float("inf") if mse == 0 else 10 * math.log10((255**2) / mse)

    snr = compute_snr(original_bytes, recovered.encode("utf-8"))
    energy_compaction = compute_energy_compaction_lzw(codes, len(data))

    return {
        "algorithm": "LZW",
        "original_size_bytes": original_size,
        "compressed_size_bytes": compressed_size,
        "compression_ratio": round(compression_ratio, 4),
        "bits_per_symbol": round(bits_per_symbol, 4),
        "entropy": round(entropy, 4),
        "redundancy": round(redundancy, 4),
        "mse": round(mse, 6),
        "psnr": round(psnr, 4) if psnr != float("inf") else "inf",
        "snr": round(snr, 4) if snr != float("inf") else "inf",
        "energy_compaction": round(energy_compaction, 4),
        "num_codes": len(codes),
        "final_dict_size": FIRST_CODE + len(codes),
    }

In [141]:
# Evaluating all the algorithms

In [142]:
DATASETS_DIR = "./datasets"
PLOTS_DIR = "./plots"
RESULTS_DIR = "./results"

DATASET_FILES = {
    "High Redundancy\n(Repeated Words)": "dataset1_high_redundancy.txt",
    "Large English\n(10000 chars)": "large_english.txt",
    "Low Redundancy\n(Random Chars)": "dataset3_low_redundancy.txt",
}

ALGOS = ["LZ77", "LZW", "Huffman", "Deflate"]

COLORS = {
    "LZ77": "#E63946",
    "LZW": "#457B9D",
    "Huffman": "#2A9D8F",
    "Deflate": "#F4A261",
}

MARKERS = {
    "LZ77": "o",
    "LZW": "s",
    "Huffman": "^",
    "Deflate": "D",
}

def load(fname):
    f = open(DATASETS_DIR + "/" + fname, "r")
    d = f.read()
    f.close()
    return d


def describe(data):
    from collections import Counter
    import math

    c = Counter(data)
    n = len(data)
    ent = 0
    for v in c.values():
        if v:
            p = v / n
            ent -= p * math.log2(p)

    return {
        "size": n,
        "unique": len(c),
        "entropy": ent,
        "redundancy": 1 - ent / 8.0,
    }


print("running...")

all_results = {}
dataset_props = {}

for name in DATASET_FILES:
    d = load(DATASET_FILES[name])
    dataset_props[name] = describe(d)

for name in DATASET_FILES:
    d = load(DATASET_FILES[name])
    all_results[name] = {}

    for algo in ALGOS:
        import time

        t0 = time.time()

        if algo == "LZ77":
            m = evaluate_lz77(d, window_size=255, lookahead_size=15)
        elif algo == "LZW":
            m = evaluate_lzw(d)
        elif algo == "Huffman":
            m = evaluate_huffman(d)
        else:
            m = evaluate_deflate(d)

        m["time_sec"] = time.time() - t0

        all_results[name][algo] = m

        print(name.replace("\n", " "), algo, m["compression_ratio"])


out = []
out.append("RESULTS\n")

for d in all_results:
    for algo in all_results[d]:
        m = all_results[d][algo]
        row = d.replace("\n", " ") + " " + algo + " "
        row += str(m["compression_ratio"]) + " "
        row += str(m["bits_per_symbol"]) + " "
        row += str(m["time_sec"])
        out.append(row)

# !!!!!!!!!!!!!!!!!!!!
# make sure that the file results_summary.txt already exists
os.makedirs(RESULTS_DIR, exist_ok=True)
f = open(RESULTS_DIR + "/results_summary.txt", "w")
f.write("\n".join(out))
f.close()

os.makedirs(PLOTS_DIR, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

names = list(all_results.keys())
short = [x.split("\n")[0] for x in names]
x = np.arange(len(names))
w = 0.25


# compression ratio
plt.figure(figsize=(10, 5))

i = 0
for algo in ALGOS:
    vals = []
    for d in names:
        vals.append(all_results[d][algo]["compression_ratio"])

    plt.bar(
        x + (i - 1) * w,
        vals,
        w,
        label=algo,
        color=COLORS[algo],
        alpha=0.85,
        edgecolor="white",
    )
    i += 1

plt.xlabel("Dataset")
plt.ylabel("Compression Ratio")
plt.xticks(x, short)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.savefig(PLOTS_DIR + "/plot1.png", dpi=150, bbox_inches="tight")
plt.close()


# bits per symbol
plt.figure(figsize=(10, 5))

i = 0
for algo in ALGOS:
    vals = []
    for d in names:
        vals.append(all_results[d][algo]["bits_per_symbol"])

    plt.bar(
        x + (i - 1) * w,
        vals,
        w,
        label=algo,
        color=COLORS[algo],
        alpha=0.85,
        edgecolor="white",
    )
    i += 1

ent = []
for d in names:
    ent.append(dataset_props[d]["entropy"])

plt.plot(x, ent, "k--o")

plt.xlabel("Dataset")
plt.ylabel("Bits Per Symbol")
plt.xticks(x, short)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.savefig(PLOTS_DIR + "/plot2.png", dpi=150, bbox_inches="tight")
plt.close()


# entropy vs compression
plt.figure(figsize=(8, 6))

for algo in ALGOS:
    xs = []
    ys = []
    for d in names:
        xs.append(dataset_props[d]["entropy"])
        ys.append(all_results[d][algo]["compression_ratio"])

    plt.plot(
        xs,
        ys,
        MARKERS[algo] + "-",
        color=COLORS[algo],
        label=algo,
    )

plt.xlabel("Entropy")
plt.ylabel("Compression Ratio")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(PLOTS_DIR + "/plot3.png", dpi=150, bbox_inches="tight")
plt.close()

# import matplotlib.pyplot as plt
# import numpy as np

# names = list(all_results.keys())
# short = [x.split("\n")[0] for x in names]
# x = np.arange(len(names))
# w = 0.25

# # compression ratio plot
# plt.figure()

# i = 0
# for algo in ALGOS:
#     vals = []
#     for d in names:
#         vals.append(all_results[d][algo]["compression_ratio"])
#     plt.bar(x + (i - 1) * w, vals, w)
#     i += 1

# plt.xticks(x, short)
# plt.savefig(PLOTS_DIR + "/plot1.png")
# plt.close()


# # bits per symbol
# plt.figure()

# i = 0
# for algo in ALGOS:
#     vals = []
#     for d in names:
#         vals.append(all_results[d][algo]["bits_per_symbol"])
#     plt.bar(x + (i - 1) * w, vals, w)
#     i += 1

# ent = []
# for d in names:
#     ent.append(dataset_props[d]["entropy"])

# plt.plot(x, ent)
# plt.savefig(PLOTS_DIR + "/plot2.png")
# plt.close()


# # entropy vs compression
# plt.figure()

# for algo in ALGOS:
#     xs = []
#     ys = []
#     for d in names:
#         xs.append(dataset_props[d]["entropy"])
#         ys.append(all_results[d][algo]["compression_ratio"])
#     plt.plot(xs, ys)

# plt.savefig(PLOTS_DIR + "/plot3.png")
# plt.close()


# print("done")
# print("plots present in ./plots/")

running...
High Redundancy (Repeated Words) LZ77 2.3507
High Redundancy (Repeated Words) LZW 4.4425
High Redundancy (Repeated Words) Huffman 2.224
High Redundancy (Repeated Words) Deflate 3.779396619329724
Large English (10000 chars) LZ77 1.2868
Large English (10000 chars) LZW 2.0202
Large English (10000 chars) Huffman 1.8169
Large English (10000 chars) Deflate 1.9043266603745468
Low Redundancy (Random Chars) LZ77 0.8145
Low Redundancy (Random Chars) LZW 1.0835
Low Redundancy (Random Chars) Huffman 1.3224
Low Redundancy (Random Chars) Deflate 0.9806084675541173
